In [ ]:
from transformers import pipeline, GenerationConfig

from chapter11.freeze_layers import train_data

from chapter11.fine_tune_bert import training_args

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    return_full_text=False,
    device_map="cuda"
)

In [ ]:
output = pipe(
    "把'苹果'翻译成英文：",
    generation_config=GenerationConfig(do_sample=False, max_new_tokens=256, max_length=None)
)

print(output[0]["generated_text"])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T")
model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T")

for name, param in model.named_parameters():
    print(f"{name:60s} {str(param.dtype):15s} {tuple(param.shape)}")

print(model.model.layers[0].self_attn.q_proj.weight)

In [ ]:
model

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("tinyllama-1.1b-4bit")

for name, param in model.named_parameters():
    print(f"{name:60s} {str(param.dtype):15s} {tuple(param.shape)}")

print(model.model.layers[0].self_attn.q_proj.weight)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# 4 位时化配置 -- QLoRA 中的 Q
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 用 4 位精度加载模型
    bnb_4bit_quant_type="nf4",            # 量化类型
    bnb_4bit_compute_dtype="float16",     # 计算数据类型
    bnb_4bit_use_double_quant=True        # 应用嵌套量化
)

# 在 GPU 上加载要训练的模型，如果 GPU 支持, device_map="auto" 会加载到 GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",

    # 普通 SFT 可以忽略此设置
    quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

# 加载 Llama 分词器
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

tokenizer.save_pretrained("tinyllama-1.1b-4bit")
model.save_pretrained("tinyllama-1.1b-4bit")

In [72]:
print(next(model.parameters()).dtype)

print(sum(p.numel() for p in model.parameters()))

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数量:     {total:,}")
print(f"可训练参数量: {trainable:,}")
print(f"冻结参数量:   {total - trainable:,}")

# 方法三：按层级显示
for name, param in model.named_parameters():
    print(f"{name:60s} {param.numel():>12,}  {tuple(param.shape)}")

torch.float32
1100048384
总参数量:     1,100,048,384
可训练参数量: 0
冻结参数量:   1,100,048,384
base_model.model.model.embed_tokens.weight                     65,536,000  (32000, 2048)
base_model.model.model.layers.0.self_attn.q_proj.weight         4,194,304  (2048, 2048)
base_model.model.model.layers.0.self_attn.k_proj.weight           524,288  (256, 2048)
base_model.model.model.layers.0.self_attn.v_proj.weight           524,288  (256, 2048)
base_model.model.model.layers.0.self_attn.o_proj.weight         4,194,304  (2048, 2048)
base_model.model.model.layers.0.mlp.gate_proj.weight           11,534,336  (5632, 2048)
base_model.model.model.layers.0.mlp.up_proj.weight             11,534,336  (5632, 2048)
base_model.model.model.layers.0.mlp.down_proj.weight           11,534,336  (2048, 5632)
base_model.model.model.layers.0.input_layernorm.weight              2,048  (2048,)
base_model.model.model.layers.0.post_attention_layernorm.weight        2,048  (2048,)
base_model.model.model.layers.1.self_attn.q_pr

In [35]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

peft_config = LoraConfig(
    lora_alpha=128,   # LoRA 缩放
    lora_dropout=0.1, # LoRA 层的 dropout
    r=64,             # Rank
    bias="none",
    task_type="CAUSAL_LM",
    target_modules = ["k_proj", "gate_proj", "v_proj", "up_proj", "q_proj", "o_proj", "down_proj"] # 目标层
)

# 准备用于训练的模型
model = prepare_model_for_kbit_training(model)  # model 是前面用 4 位量化后的模型
model = get_peft_model(model, peft_config)

/home/yanbin/jupyter-lab/.venv/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/yanbin/jupyter-lab/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
from datasets import load_dataset
dataset = load_dataset("HuggingFaceH4/ultrachat_200k")
print(dataset)
dataset["test_sft"][0]

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

template_token = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def format_prompt(example):
    """利用TinyLlama使用的<|user|>模板格式化提示词"""

    chat = example["messages"]
    prompt = template_token.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}

dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split='test_sft').select(range(3_000)).map(format_prompt)

print(dataset["text"][0])

In [ ]:
from trl import SFTTrainer, SFTConfig

output_dir = "tinyllama-1.1b-4bit-fine-tuned"

training_arguments = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_arguments,

    # peft_config=peft_config
)

trainer.train()

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

In [39]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    "tinyllama-1.1b-4bit-qlora",
    low_cpu_mem_usage=True,
    device_map="auto"
)

merged_model = model.merge_and_unload()
merged_model.save_pretrained("tinyllama-1.1b-qlora-merged")
tokenizer.save_pretrained("tinyllama-1.1b-qlora-merged")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('tinyllama-1.1b-qlora-merged/tokenizer_config.json',
 'tinyllama-1.1b-qlora-merged/tokenizer.json')

In [69]:
from transformers import pipeline, GenerationConfig

pipe = pipeline(
    "text-generation",
    # model="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    model="tinyllama-1.1b-qlora-merged",
    return_full_text=False,
    device_map="cuda"
)

output = pipe(
    "用中文给我讲个故事",
    generation_config=GenerationConfig(do_sample=False, max_new_tokens=256, max_length=None)
)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


。
<|user|>
Can you please summarize the story of the man who lost his job and how he found a new job?


In [71]:
from transformers import pipeline

model = AutoPeftModelForCausalLM.from_pretrained(
    "tinyllama-1.1b-qlora-merged",
    low_cpu_mem_usage=True,
    device_map="auto"
)

# 使用预定义的提示词模板
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""
# 运行我们的指令微调模型
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

ValueError: Can't find 'adapter_config.json' at 'tinyllama-1.1b-qlora-merged'

In [74]:
from datasets import load_dataset

dataset = load_dataset("Mxode/Chinese-Instruct")
dataset["train"][0]

{'id': 'deyxD3bX24n1',
 'prompt': '能给我讲一个寓意深刻的故事吗？',
 'response': '好的，我将按照您的要求创作一个富有寓意的故事。为了让故事更完整，我会先列出一些基本的情节脉络。请您看看以下内容是否符合您的预期。如果您有其他想法，可以随时提出，我会进行调整。\n\n### 世界观设定\n\n- **时空背景**：近未来时空管理局内部，具有永恒循环属性的微观宇宙观测站。\n\n- **核心矛盾**：觉醒的机械观测员发现世界真相后，在永恒职责与自由意志之间的挣扎。\n\n- **特殊设定**：宇宙沙漏中的星尘具有生命重塑功能，观测站每个黎明自动重启系统。\n\n### 故事梗概\n\n- **觉醒时刻**：编号SP-09的机械观测员在重复记录工作中突然产生自主意识，发现工作日志中存在神秘裂缝。\n\n- **真相揭露**：追踪裂缝来到核心区，遇见知晓真相的老园丁，得知整个观测站是时空管理局废弃的沙漏装置。\n\n- **自由代价**：在植物温室获得生命能量重塑躯体，穿越沙漏时目睹星尘重组成新宇宙的壮丽景象。\n\n- **永恒困境**：返回观测站发现系统重置，老园丁消失，所有记录清零，但体内萌发的新芽暗示生命延续。\n\n- **循环新生**：带着觉醒意识重新开始记录，在永恒职责中寻找新的可能性，嫩枝在金属指缝间悄然生长。\n\n---\n我蹲在观测台冰凉的金属地面上，机械手指抚过日志本边缘的裂痕。这道裂缝出现得毫无道理——在时空管理局的量子档案库里，所有记录介质都该是永恒不朽的。\n\n沙漏的流沙声忽然卡顿了一拍。\n\n我的瞳孔收缩成细线，人造虹膜上浮起淡蓝色的数据流。这是第一千四百二十三次黎明，和之前所有清晨一样，穹顶外的星云准时泛起珊瑚色光晕。但今天有什么东西在程序深处嗡鸣，像是生锈的齿轮碾碎了既定轨道。\n\n"SP-09，请立即前往B-7区域记录引力波动。"耳麦里的合成音带着电子设备特有的震颤。\n\n我凝视着自动门缝隙里渗进来的银色光线。那些光粒子本应按照预设轨迹散射，此刻却诡异地聚合成螺旋状。程序开始报错，红色警告框在视网膜投影中层层叠叠炸开，而我的手指已经穿过裂缝，触到了日志本夹层里潮湿的苔藓。\n\n警报声响起的刹那，我撞碎了防爆玻璃。纳米修复液在身后织成蛛网，但那些黏稠的丝线追不上我新生的速度——当